<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Analyze large scenario batches and produce campaign-level comparisons.
- **Primary inputs**: runs/many_scenarios/<run_id>/*/output.csv + runs/many_scenarios/<run_id>/policies/indicator.csv
- **Primary outputs**: derived comparison tables and campaign figures
- **Refactor role**: Keep as scenario-analysis notebook. Reuse shared run-loading helpers to reduce duplicated IO code.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Load scenario outputs and policy indicators for a selected run.
2. Create grouped policy-package views (e.g., market failures, policy package variants).
3. Build comparison visuals and summary tables for selected outcomes.

### Inputs
- `runs/many_scenarios/<run_id>/*/output.csv`
- `runs/many_scenarios/<run_id>/policies/indicator.csv`

### Outputs
- Scenario grouping tables and charts in the notebook output folder.
- `indicator.csv` in configured output folder when enabled in notebook logic.

### How To Reuse
- Update `folder, assessment = ...` to the run id and assessment mode you want.


#  Analysis of the results
This notebook is used to analyze the results of simulations that have been run with full combination of policies.
It is used to create figures for the report.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from project.write_output import plot_compare_scenarios_simple

from project.write_output import plot_compare_scenarios, indicator_policies, make_summary
from project.utils import get_json, manual_sobol_analysis, horizontal_stack_bar_plot, manual_shapley_analysis
from project.utils import format_ax, format_legend

import itertools

import warnings
warnings.filterwarnings('ignore')

## Loading data

In [ ]:
folder, assessment = 'policies_scenarios_20240528_203907', "market_failures"
#folder, assessment = 'policies_scenarios_20240529_101722', "optimal_pp"
# folder, assessment = 'policies_scenarios_20240522_102305', "policies"

scenarios_description = 'policies_scenarios_description.csv'


# assessment = "policies"
# assessment = "market_failures"
# assessment = "optimal_pp"

# Refactor: run data is stored under scenario_analysis/runs/many_scenarios
folder = os.path.join('runs', 'many_scenarios', folder)
folder_out = os.path.join(folder, 'img')
if not os.path.exists(folder_out):
    os.makedirs(folder_out)


In [ ]:
# Read scenarios description and convert all columns to str
scenarios_description = pd.read_csv(os.path.join(folder, scenarios_description), index_col=[0]).rename_axis('Scenarios')
list_features = list(scenarios_description.columns)

for col in list_features:
    scenarios_description[col] = scenarios_description[col].astype(str)

In [ ]:
# In folder open all child folders and read the output.csv files
dict_output = {child_folder: pd.read_csv(os.path.join(folder, child_folder, 'output.csv'), header=[0], index_col=[0])
             for child_folder in os.listdir(folder)
             if os.path.isdir(os.path.join(folder, child_folder))
             and os.path.isfile(os.path.join(folder, child_folder, 'output.csv'))}

# convert all columns to int
for scenario in dict_output.keys():
    dict_output[scenario].columns = [int(c) for c in dict_output[scenario].columns]

In [ ]:
reference = 'Reference'
# Exogenous parameters used to assess policies
config_policies = get_json('project/input/policies/cba_inputs.json')
order_scenarios = None
_ = indicator_policies(dict_output, folder, config_policies, reference=reference, figure=False)
data = pd.read_csv(os.path.join(folder, 'policies/indicator.csv'), index_col=[0])
data = pd.concat((scenarios_description, data.T), axis=1)
data.to_csv(os.path.join(folder_out, 'indicator.csv'))

In [ ]:
# remove some scenarios
if True and assessment == 'optimal_pp':
    to_drop = ['subsidy_status_quo', 'subsidy_present_bias']
    to_drop = [c for c in to_drop if c in data.columns]
    for c in to_drop:
        data = data.loc[data[c] != c, :]
        data.drop(columns=c, inplace=True)
        list_features = [f for f in list_features if f != c]
    

In [ ]:
data['Group'] = 'Other'

# select row where all features == 'no_{}'.format(feature)
idx = data.index
for feature in list_features:
    idx = idx.intersection(data[data[feature].str.split('_').str[0] == 'no'].index)
if assessment == 'market_failures':
    data.loc[idx, 'Group'] = 'All friction'
elif assessment in ['policies', 'optimal_pp']:
    data.loc[idx, 'Group'] = 'No policy'
else:
    pass

idx = data.index
for feature in list_features:
    idx = idx.intersection(data[data[feature].str.split('_').str[0] != 'no'].index)
if assessment == 'market_failures':
    data.loc[idx, 'Group'] = 'No friction'
elif assessment == 'optimal_pp':
    data.loc[idx, 'Group'] = 'Second best subsidies'
    data = data[data['subsidy_emission'] != 'carbon_tax_vsc']
else:
    pass

# standalone effect
for feature in list_features:
    for f in data[feature].dropna().unique():
        idx = data.index
        if f.split('_')[0] != 'no':
            idx = idx.intersection(data[data[feature] == f].index)
            for of in [k for k in list_features if k != feature]:
                idx = idx.intersection(data[data[of].str.split('_').str[0] == 'no'].index)
            data.loc[idx, 'Group'] = f


additional_scenarios = {
    'CarbonTax': 'SCC Benchmark',
    'CurrentPolicies': 'Package 2018',
    'Package2021': 'Package 2021',
    'Package2024': 'Baseline Package',
    'Package2024Ban': 'Baseline Package + Ban',
    'NoMF': 'No friction'
}
for k, i in additional_scenarios.items():
    data.loc[data.index == k, 'Group'] = i


if False:
    packages_policies = {
        'P2018': {'subsidies': 'subsidies_2018', 'restriction_gas': 'no_restriction_gas', 'carbon_tax': 'carbon_tax',
                         'obligation': 'no_obligation', 'cee': 'cee_2018', 'zero_interest_loan': 'no_zero_interest_loan'},
        'P2021': {'subsidies': 'subsidies_2021', 'restriction_gas': 'no_restriction_gas', 'carbon_tax': 'carbon_tax',
                            'obligation': 'obligation', 'cee': 'cee_2021', 'zero_interest_loan': 'no_zero_interest_loan'},
        'P2024': {'subsidies': 'subsidies_2024', 'restriction_gas': 'no_restriction_gas', 'carbon_tax': 'carbon_tax_high',
                            'obligation': 'obligation', 'cee': 'cee_2024', 'zero_interest_loan': 'zero_interest_loan'},
        'P2024 + Ban': {'subsidies': 'subsidies_2024', 'restriction_gas': 'restriction_gas', 'carbon_tax': 'carbon_tax_high',
                            'obligation': 'obligation', 'cee': 'cee_2024', 'zero_interest_loan': 'zero_interest_loan'}
    }
    
    idx = data.index
    
    if assessment == 'policies':
        for package, policies in packages_policies.items():
            idx = data.index
            for feature, value in policies.items():
                idx = idx.intersection(data[data[feature] == value].index)
            data.loc[idx, 'Group'] = package
        
data.loc[:, 'Size'] = 100
data.loc[data['Group'] != 'Other', 'Size'] = 200

In [ ]:
# need to add district heating emission and electricity in the scope
objective_carbon_neutrality = 523
emission_reference = dict_output[reference].loc['Emission (MtCO2)'].sum()
emission_saving_objective = emission_reference - objective_carbon_neutrality

energy_saving_objective = None
emission_saving_objective = None

# TODO
- Add reference scenario (no policy scenario)
- Add 2018, 2021 and 2014 policy scenarios
- Add first-best, second-best, etc...
- Add optimum scenario
- Merge les credit constraint heater & insulation

## Formatting data

In [ ]:
dict_replace = {
    'ghg_externality': 'GHG externalities',
    #'no_ghg_externality': 'GHG externalities',
    'carbon_tax_vsc': 'SCC Benchmark',
    'carbon_tax_high': 'Carbon tax',
    'carbon_tax': 'Carbon tax',
    'subsidy_emission': 'Subsidies emission',
    'subsidy_health_cost': 'Subsidies health cost',
    'subsidy_landlord': 'Subsidies landlord',
    'subsidy_multi_family': 'Subsidies multi family',
    'subsidy_present_bias': 'Subsidies present bias',
    'subsidy_status_quo': 'Subsidies status quo',
    'tax_status_quo': 'Tax status quo',
    'landlord_dilemma': 'Landlord dilemma',
    # 'no_landlord_dilemma': 'No landlord dilemma',
    'multi_family': 'Multi family friction',
    'health_cost': 'Health cost',
    # 'no_multi_family': 'No multi family friction',
    'credit_constraint': 'Credit constraint',
    'credit_constraint_heater': 'Credit constraint heater',
    'credit_constraint_insulation': 'Credit constraint insulation',
    # 'no_credit_constraint_heater': 'No credit constraint heater',
    # 'no_credit_constraint_insulation': 'No credit constraint insulation',
    'present_bias': 'Present bias',
    # 'no_present_bias': 'No present bias',
    'status_quo_bias': 'Status quo bias',
    # 'no_status_quo_bias': 'No status quo bias',
    'subsidies': 'Subsidies', 
    'subsidies_2018': 'Subsidies 2018',
    'subsidies_2021': 'Subsidies 2021',
    'subsidies_2024': 'Subsidies 2024',
    'restriction_gas': 'Ban gas', 
    'cee': 'WCO', 
    'cee_2018': 'WCO 2018',
    'cee_2021': 'WCO 2021',
    'cee_2024': 'WCO 2024',
    'zero_interest_loan': 'Zero interest loan', 
    'obligation': 'Rental ban'
}
data = data.replace(dict_replace)

color_dict = {
    'GHG externalities': 'darkgreen',
    'Subsidies emission': 'darkgreen',
    'Carbon tax': 'darkgreen',
    'Carbon tax high': 'darkgreen',
    'Carbon tax + EU-ETS 2': 'darkgreen',
    'SCC Benchmark': 'darkgreen',
    'Landlord dilemma': 'orange',
    'Subsidies landlord': 'orange',
    'Multi family friction': 'darkmagenta',
    'Subsidies multi family': 'darkmagenta',
    'Credit constraint': 'red',
    'Credit constraint insulation': 'red',
    'Credit constraint heater': 'red',
    'Present bias': 'blue',
    'Subsidies present bias': 'blue',
    'Status quo bias': 'purple',
    'Subsidies status quo': 'purple',
    'Tax status quo': 'purple',
    'Subsidies health cost': 'yellow',
    'Other': 'lightgrey',
    'No policy': 'black',
    'All friction': 'black',
    'No friction': 'green',
    'Second best subsidies': 'green',
    'Health cost': 'yellow',
    'Subsidies': 'royalblue',
    'Subsidies 2018': 'royalblue',
    'Subsidies 2021': 'royalblue',
    'Subsidies 2024': 'royalblue',
    'Ban gas': 'red', 
    'WCO': 'darkorange',
    'WCO 2018': 'darkorange',
    'WCO 2021': 'darkorange',
    'WCO 2024': 'darkorange',
    'White certificate obligation': 'orange', 
    'Zero interest loan': 'darkmagenta', 
    'Rental ban': 'firebrick',
    'Package 2018': 'royalblue',
    'Package 2021': 'darkorange',
    'Baseline Package': 'firebrick',
    'Baseline Package + Ban': 'darkmagenta',
    'P2018': 'royalblue',
    'P2021': 'darkorange',
    'P2024': 'firebrick',
    'P2024 + Ban': 'darkmagenta',
    'First best': 'green',
    'Second best': 'blue',
    'Third best': 'red',
    'Carbon tax marginal damage': 'black',
    'Combination': 'grey'
}

## Mapping energy efficiency gap

In [ ]:
def mapping(x='NPV annual (Billion euro/year)', y='Cumulated energy saving (TWh)', hue='Group', file=os.path.join(folder_out, 'figure.pdf'),
            format_y=lambda k, _: '{:.0f}'.format(k), format_x=lambda k, _: '{:.0f}'.format(k), annotate=None,
            x_label=None, remove_legend=None, xmin=None, xmax=None, legend_title='', size=None, hline=None, ymax=None):
    
    annotate = [i for i in annotate if i in data[hue].values]
    
    if size is None:
        size = 100
        df = data.loc[:, [x, y, hue]]
        df.sort_values(hue, inplace=True)
    else:
        df = data.loc[:, [x, y, hue, size]].dropna()
        df.sort_values(hue, inplace=True)
        size = df[size]

    
    fig, ax = plt.subplots(figsize=(12.8, 9.6))
    
    sns.scatterplot(
        data=df, x=x, y=y,
        ax=ax, s=size, hue=hue, palette=color_dict, style=hue
    )
    # sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


    
    format_ax(ax, title=y, format_y=format_y, y_label='',
              format_x=format_x, ymin=None, xmin=xmin, xmax=xmax, ymax=ymax)
    if x_label is not None:
        ax.set_xlabel(x_label)
    
    y_max = df[y].max()
    if hline is not None:
        y_max = max(df[y].max(), hline)
    
    margin_x = (df[x].max() - df[x].min()) * 0.1
    margin_y = (y_max - df[y].min()) * 0.1
    ax.set_xlim([df[x].min() - margin_x, df[x].max() + margin_x])
    ax.set_ylim([df[y].min() - margin_y, y_max + margin_y])
    
    ax.axhline(0, color='black', lw=1)
    if hline is not None:
        ax.axhline(hline, color='red', lw=1, linestyle='--')

    if annotate is not None:
        for k in annotate:
            if k in df[hue].values:
                position = (df[df[hue] == k][x], df[df[hue] == k][y])
                ax.annotate(k, position, fontsize=12, color=color_dict[k], fontweight='bold')
    
    
    # remove annotate from legend
    handles, labels = ax.get_legend_handles_labels()
    
    if annotate is not None:
        temp = annotate
        if remove_legend is not None:
            print()
            temp += remove_legend
    
        new_legend = [(h, l) for h, l in zip(handles, labels) if l not in temp]
        handles, labels = zip(*new_legend)
        
    # color legend font with color_dict
    ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1, 1), frameon=False, title=legend_title)
     
    #
    #plt.show()
    fig.savefig(file, bbox_inches='tight')

    plt.close(fig)
    
    return None
        

In [ ]:
if assessment == 'policies':
    # energy gap
    annotate = ['Package 2018', 'Package 2021', 'Baseline Package', 'Baseline Package + Ban', 'No policy', 'SCC Benchmark', 'No friction']
    annotate += ['P2018', 'P2021', 'P2024', 'P2024 + Ban']
    remove_legend = ['Other']
    x_label =  'Social welfare compared to No policy (billion euro per year)'
    mapping(x='NPV annual (Billion euro/year)', y='Cumulated energy saving (TWh)', hue='Group', file=os.path.join(folder_out, 'mapping_energy_efficiency_gap_policies.pdf'),
            format_y=lambda k, _: '{:.0f}'.format(k), format_x=lambda k, _: '{:.0f}'.format(k), annotate=annotate,
            x_label=x_label, remove_legend=remove_legend, legend_title='Policies standalone', size='Size', hline=energy_saving_objective)
    
    # emission gap
    annotate = ['Package 2018', 'Package 2021', 'Baseline Package', 'Baseline Package + Ban', 'No policy', 'SCC Benchmark', 'No friction']
    remove_legend = ['Other']
    x_label =  'Social welfare compared to No policy (billion euro per year)'
    mapping(x='NPV annual (Billion euro/year)', y='Cumulated emission saving (MtCO2)', hue='Group', file=os.path.join(folder_out, 'mapping_emission_efficiency_gap_policies.pdf'),
            format_y=lambda k, _: '{:.0f}'.format(k), format_x=lambda k, _: '{:.0f}'.format(k), annotate=annotate,
            x_label=x_label, remove_legend=remove_legend, legend_title='Policies standalone', size='Size',
            hline=emission_saving_objective)
    
    annotate = ['Package 2018', 'Package 2021', 'Baseline Package', 'Baseline Package + Ban', 'No policy', 'SCC Benchmark', 'No friction']
    remove_legend = ['Other']
    x_label =  'Social welfare compared to No policy (billion euro per year)'
    mapping(x='Investment / energy savings (euro/kWh)', y='Cumulated energy saving (TWh)', hue='Group',
            file=os.path.join(folder_out, 'mapping_cost_efficiency_kwh_policies.pdf'),
            format_y=lambda k, _: '{:.0f}'.format(k), format_x=lambda k, _: '{:.2f}'.format(k), annotate=annotate,
            x_label=x_label, remove_legend=remove_legend, legend_title='Policies standalone', size='Size',
            hline=emission_saving_objective)
    
    annotate = ['Package 2018', 'Package 2021', 'Baseline Package', 'Baseline Package + Ban', 'No policy', 'SCC Benchmark', 'No friction']
    remove_legend = ['Other']
    x_label =  'Investment / emission (euro/tCO2)'
    mapping(x='Investment / emission (euro/tCO2)', y='Cumulated emission saving (MtCO2)', hue='Group',
            file=os.path.join(folder_out, 'mapping_cost_efficiency_tCO2_policies.pdf'),
            format_y=lambda k, _: '{:.0f}'.format(k), format_x=lambda k, _: '{:.0f}'.format(k), annotate=annotate,
            x_label=x_label, remove_legend=remove_legend, legend_title='Policies standalone', size='Size',
            hline=emission_saving_objective)


elif assessment == 'market_failures':
    annotate = ['No friction', 'All friction']
    remove_legend = ['Other']
    x_label =  'Social welfare compared to No friction (billion euro per year)'
    
    
    mapping(x='NPV annual (Billion euro/year)', y='Cumulated energy saving (TWh)', hue='Group', file=os.path.join(folder_out, 'mapping_energy_efficiency_gap_mf.pdf'),
            format_y=lambda k, _: '{:.0f}'.format(k), format_x=lambda k, _: '{:.0f}'.format(k), annotate=annotate,
            x_label=x_label, remove_legend=remove_legend, legend_title='Market failures standalone', size='Size',
            hline=energy_saving_objective)
    
    annotate = ['No friction', 'All friction']
    remove_legend = ['Other']
    # emission gap
    mapping(x='NPV annual (Billion euro/year)', y='Cumulated emission saving (MtCO2)', hue='Group', file=os.path.join(folder_out, 'mapping_emission_efficiency_gap_mf.pdf'),
            format_y=lambda k, _: '{:.0f}'.format(k), format_x=lambda k, _: '{:.0f}'.format(k), annotate=annotate,
            x_label=x_label, remove_legend=remove_legend, legend_title='Market failures standalone', size='Size',
            hline=emission_saving_objective)

elif assessment == 'optimal_pp':
    annotate = ['No policy', 'Second best subsidies', 'Baseline Package', 'No friction']
    x_label =  'Cost-benefits analysis (billion euro per year)'
    remove_legend = ['Other']
    mapping(x='NPV annual (Billion euro/year)', y='Cumulated energy saving (TWh)', hue='Group', file=os.path.join(folder_out, 'mapping_energy_efficiency_gap_optimal_pp.pdf'),
        format_y=lambda k, _: '{:.0f}'.format(k), format_x=lambda k, _: '{:.0f}'.format(k), annotate=annotate,
        x_label=x_label, remove_legend=remove_legend, legend_title='Policies standalone', size='Size',
            hline=energy_saving_objective)
    
    annotate = ['No policy', 'Second best subsidies', 'Baseline Package', 'No friction']
    x_label =  'Cost-benefits analysis (billion euro per year)'
    remove_legend = ['Other']
    mapping(x='NPV annual (Billion euro/year)', y='Cumulated emission saving (MtCO2)', hue='Group', file=os.path.join(folder_out, 'mapping_emission_efficiency_gap_optimal_pp.pdf'),
            format_y=lambda k, _: '{:.0f}'.format(k), format_x=lambda k, _: '{:.0f}'.format(k), annotate=annotate,
            x_label=x_label, remove_legend=remove_legend, legend_title='Policies standalone', size='Size',
            hline=emission_saving_objective)
    


## Measuring interactions

In [ ]:
def interaction(key='NPV annual (Billion euro/year)'):
    rslt = {}
    # iteration over policy groups
    for feature in list_features:
        # iteration over policy
        for policy in [i for i in data[feature].unique() if i is not np.nan]:
            if policy.split('_')[0] != 'no':
                # iteration over all interactions with other policies
                for n, g in data.groupby([k for k in list_features if k != feature]):
                    if key == 'NPV annual (Billion euro/year)':
                        val = float(g.loc[g[feature] == policy, key]) - float(g.loc[g[feature] == 'no_{}'.format(feature), key])
                    elif key in ['Investment / emission (euro/tCO2)', 'Investment / energy savings (euro/kWh)']:
                        num = 'Investment total WT (Billion euro)'
                        num = float(g.loc[g[feature] == policy, num]) - float(g.loc[g[feature] == 'no_{}'.format(feature), num])
                        if key == 'Investment / emission (euro/tCO2)':
                            den = 'Emission (MtCO2)'
                        elif key == 'Investment / energy savings (euro/kWh)':
                            den = 'Consumption (TWh)'
                        else:
                            raise ValueError('key not recognized')
                        den = - (float(g.loc[g[feature] == policy, den]) - float(g.loc[g[feature] == 'no_{}'.format(feature), den]))
                        val = num / den
                        if key == 'Investment / emission (euro/tCO2)':
                            val *= 1e3

                    else:
                        raise ValueError('key not recognized')
                    name = n
                    if len([x for x in n if 'no' == x.split('_')[0]]) == len(n):
                        name = 'No policy'
                    if len([x for x in n if 'no' == x.split('_')[0]]) == len(n) - 1:
                        name = [x for x in n if 'no' != x.split('_')[0]][0]
                    rslt.update({(policy, name): val})

    rslt = pd.Series(rslt)
    rslt = rslt.reset_index().set_axis(['policy', 'group'
                                                  '', 'value'], axis=1)
    rslt['hue'] = 'Combination'
    rslt.loc[rslt['group'] == 'No policy', 'hue'] = 'No policy'

    return rslt

def make_interation_heatmap(temp, file, fmt=".1f", xlabel='Social welfare (billion euro per year)', orders=None):
    fig, ax = plt.subplots(figsize=(12.8, 9.6))

    glue = temp.pivot(index="policy", columns="group", values="value")

    if orders is not None:
        orders = [i for i in orders if i in glue.index] + [i for i in glue.index if i not in orders]
        glue = glue.loc[orders, orders]

    cmap = sns.diverging_palette(220, 20, as_cmap=True, center="light")

    sns.heatmap(glue, annot=False, fmt=fmt, ax=ax, cbar_kws={'label': xlabel}, center=0, cmap=cmap) #cmap='coolwarm',

    # Manually add annotations with conditional bold font weight
    for i, row in enumerate(glue.index):
        for j, col in enumerate(glue.columns):
            value = glue.at[row, col]
            if not np.isnan(value):  # Only add text if value is not NaN
                ax.text(j+0.5, i+0.5, format(value, fmt),
                        ha='center', va='center',
                        fontweight='bold' if row == col else 'normal')  # Apply bold if row index matches column label


    ax.set_xlabel('In interaction with')
    ax.set_ylabel('Policies')
    # put x axis at top
    ax.xaxis.tick_top()
    # rotation of x ticks
    plt.xticks(rotation=90)
    # put x label at top
    ax.xaxis.set_label_position('top')

    fig.savefig(file, bbox_inches='tight')
    plt.close(fig)

def make_interation_heatmap_percent(temp, file, xlabel='Social welfare (billion euro per year)', orders=None):
    fig, ax = plt.subplots(figsize=(12.8, 9.6))

    glue = temp.pivot(index="policy", columns="group", values="value")

    if orders is not None:
        orders = [i for i in orders if i in glue.index] + [i for i in glue.index if i not in orders]
        glue = glue.loc[orders, orders]

    cmap = sns.diverging_palette(20, 150, as_cmap=True, center="light")  # Custom colormap for positive (green) and negative (red)

    # Calculate relative differences for off-diagonal elements
    for i in glue.index:
        for j in glue.columns:
            if i != j:
                glue.at[i, j] = ((glue.at[i, j] - glue.at[i, i]) / glue.at[i, i]) * 100

    sns.heatmap(glue, annot=False, fmt=".1f", ax=ax, cbar=False, center=0, cmap=cmap,
                mask=np.eye(len(glue), dtype=bool))  # Mask the diagonal for color

    for i, row in enumerate(glue.index):
        for j, col in enumerate(glue.columns):
            value = glue.at[row, col]
            if not np.isnan(value):
                if row == col:
                    ax.text(j+0.5, i+0.5, f"{value:.1f}",
                            ha='center', va='center', fontweight='bold', color='black')
                else:
                    ax.text(j+0.5, i+0.5, f"{value:.0f}%",
                            ha='center', va='center', color='black')

    ax.set_xlabel('In interaction with')
    ax.set_ylabel('Policies')
    ax.xaxis.tick_top()
    plt.xticks(rotation=90)
    ax.xaxis.set_label_position('top')

    fig.savefig(file, bbox_inches='tight')
    plt.close(fig)


def make_swarmplot(df, file, xlabel='Social welfare (billion euro per year)', orders=None, xmin=None, xmax=None, format_x=lambda x, _: '{:.1f}'.format(x)):
    fig, ax = plt.subplots(figsize=(12.8, 9.6))

    if orders is not None:
        orders = [i for i in orders if i in rslt['policy'].unique()] + [i for i in rslt['policy'].unique() if i not in orders]
        df = df.sort_values(by='policy', key=lambda x: x.map({k: i for i, k in enumerate(orders)}))
    sns.swarmplot(ax=ax, data=df, x="value", y="policy", hue="hue", size=5)

    if xmin is not None and xmax is not None:
        ax.set_xlim(xmin, xmax)


    #format_ax(ax, format_x=lambda k, _: '{:.2f}'.format(k), xmin=xmin, xmax=xmax)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')
    ax.get_legend().remove()
    fig.savefig(file, bbox_inches='tight')
    plt.close(fig)

In [ ]:
orders = None
orders_rslt = None
if assessment == 'policies':
    orders = ['Carbon tax', 'Carbon tax + EU-ETS 2', 'Subsidies 2018', 'Subsidies 2021', 'Subsidies 2024', 'WCO 2018', 'WCO 2021', 'WCO 2024', 'Zero interest loan', 'Rental ban', 'Ban gas']


rslt = interaction('NPV annual (Billion euro/year)')

make_swarmplot(rslt, os.path.join(folder_out, 'assessment_swarmplot_npv.pdf'),  xlabel='Social welfare', orders=orders)

# only select str element in group from rslt
temp = rslt[rslt['group'].apply(lambda x: isinstance(x, str))]
# when no policy is present, the group is the policy
temp.loc[temp['group'] == 'No policy', 'group'] = temp.loc[temp['group'] == 'No policy', 'policy']
make_interation_heatmap(temp, os.path.join(folder_out, 'interactions_heatmap_npv.pdf'), fmt=".2f", orders=orders)
make_interation_heatmap_percent(temp, os.path.join(folder_out, 'interactions_heatmap_percent_npv.pdf'), orders=orders)


rslt = interaction('Investment / emission (euro/tCO2)')

make_swarmplot(rslt, os.path.join(folder_out, 'assessment_swarmplot_cost_efficiency_tco2.pdf'), xlabel='Investment / emission avoided (euro/tCO2)', orders=orders, xmin=0, xmax=2000)
# only select str element in group from rslt
temp = rslt[rslt['group'].apply(lambda x: isinstance(x, str))]
# when no policy is present, the group is the policy
temp.loc[temp['group'] == 'No policy', 'group'] = temp.loc[temp['group'] == 'No policy', 'policy']
make_interation_heatmap(temp, os.path.join(folder_out, 'interactions_cost_efficiency_tco2.pdf'), fmt=".0f", xlabel='Investment / emission avoided (euro/tCO2)', orders=orders)

rslt = interaction('Investment / energy savings (euro/kWh)')

make_swarmplot(rslt, os.path.join(folder_out, 'assessment_swarmplot_cost_efficiency_kwh.pdf'), xlabel='Investment / energy savings (euro/kWh)', orders=orders, xmin=0, xmax=1)

# only select str element in group from rslt
temp = rslt[rslt['group'].apply(lambda x: isinstance(x, str))]
# when no policy is present, the group is the policy
temp.loc[temp['group'] == 'No policy', 'group'] = temp.loc[temp['group'] == 'No policy', 'policy']

make_interation_heatmap(temp, os.path.join(folder_out, 'interactions_cost_efficiency_kwh.pdf'), fmt=".2f", xlabel='Investment / energy savings (euro/kWh)', orders=orders)


## Sobol analysis

In [ ]:
outcomes = ['Cumulated emission saving (MtCO2)', 'Cumulated energy saving (TWh)', 'NPV annual (Billion euro/year)']
NAME_COLUMNS = None

l_features = list_features
if False:
    # removing ban gas
    if 'restriction_gas' in data.columns:
        df = data[data['restriction_gas'] == 'no_restriction_gas']
        l_features = [i for i in list_features if i != 'restriction_gas']
    else:
        df = data.copy()

df = data.copy()
if assessment == 'policies':
    rename_columns = {'subsidies': 'Subsidies', 'restriction_gas': 'Ban gas', 'cee': 'WCO', 'zero_interest_loan': 'Zero interest loan', 'obligation': 'Rental ban', 'carbon_tax': 'Carbon tax'}
    l_features = [rename_columns[i] for i in list_features]
    df = df.rename(columns=rename_columns)

for outcome in outcomes:
    sobol_df = manual_sobol_analysis(df, list(l_features), outcome)
    if NAME_COLUMNS is not None:
        sobol_df = sobol_df.rename(index=NAME_COLUMNS)
    else:
        sobol_df.index = sobol_df.index.map(lambda x: x.replace('_', ' ').capitalize())
        
        
    outcome_name = outcome.replace(' ', '_').split('(')[0]
    
    if assessment == 'market_failures':
        title = 'Most influencial market-failures on {}'.format(outcome.split('(')[0])
    elif assessment in ['optimal_pp', 'policies']:
        title = 'Most influencial policies on {}'.format(outcome.split('(')[0])

    horizontal_stack_bar_plot(sobol_df, columns=['First order', 'Total order'],
                              title=title, order='Total order',
                              save_path=os.path.join(folder_out, 'sobol_{}.pdf'.format(outcome_name)))

## Shapley analysis

In [ ]:
print(df.head())

In [ ]:
outcomes = ['Cumulated emission saving (MtCO2)', 'Cumulated energy saving (TWh)', 'NPV annual (Billion euro/year)']
NAME_COLUMNS = None

l_features = list_features
if False:
    if 'restriction_gas' in data.columns:
        df = data[data['restriction_gas'] == 'no_restriction_gas']
        l_features = [i for i in list_features if i != 'restriction_gas']
    else:
        df = data.copy()

df = data.copy()
if assessment == 'policies':
    rename_columns = {'subsidies': 'Subsidies', 'restriction_gas': 'Ban gas', 'cee': 'WCO',
                      'zero_interest_loan': 'Zero interest loan', 'obligation': 'Rental ban',
                      'carbon_tax': 'Carbon tax'}
    l_features = [rename_columns[i] for i in list_features]
    df = df.rename(columns=rename_columns)

for outcome in outcomes:
    shapley_df = manual_shapley_analysis(df, list(l_features), outcome)
    if NAME_COLUMNS is not None:
        shapley_df = shapley_df.rename(index=NAME_COLUMNS)
    else:
        shapley_df.index = shapley_df.index.map(lambda x: x.replace('_', ' ').capitalize())

    outcome_name = outcome.replace(' ', '_').split('(')[0]
    if assessment == 'market_failures':
        title = 'Most influencial market-failures on {}'.format(outcome.split('(')[0])
    elif assessment in ['optimal_pp', 'policies']:
        title = 'Most influencial policies on {}'.format(outcome.split('(')[0])

    horizontal_stack_bar_plot(shapley_df, columns=['Shapley share'],
                              title=title, order='Shapley share',
                              save_path=os.path.join(folder_out, 'shapley_{}.pdf'.format(outcome_name)))

## Specific scenario

In [ ]:
if assessment == 'market_failures':
    reference = 'All friction'
elif assessment in ['policies', 'optimal_pp']:
    reference = 'No policy'

scenarios = data.index[data['Group'] != 'Other']

# in folder open all child folders and read the output.csv files
dict_output_subset = {k: i for k, i in dict_output.items() if k in scenarios}

# convert all columns to int
for scenario in dict_output_subset.keys():
    dict_output_subset[scenario].columns = [int(c) for c in dict_output_subset[scenario].columns]
    
temp = {data.loc[k, 'Group']: item for k, item in dict_output_subset.items()}
config_policies = get_json('project/input/policies/cba_inputs.json')
order_scenarios = None
if assessment == 'policies':
    exclude = ['Package 2018', 'Package 2021', 'Baseline Package', 'Baseline Package + Ban', 'P2018', 'P2021', 'P2024', 'P2024 + Ban', 'No friction', 'SCC Benchmark']
    temp = {k: i for k, i in temp.items() if k not in exclude}
    orders = ['Carbon tax', 'Subsidies 2018', 'Subsidies 2021', 'Subsidies 2024', 'WCO 2018', 'WCO 2021', 'WCO 2024', 'Zero interest loan', 'Rental ban', 'Ban gas']
    order_scenarios = [k for k in orders if k in temp.keys()]#[::-1]
    # reverse order

    # pp = ['Package 2018', 'Package 2021', 'Baseline Package', 'Baseline Package + Ban']
    # order_scenarios = pp + [k for k in temp.keys() if k not in pp]
    reference = 'No policy'
    
if assessment == 'market_failures':
    remove = ['No friction']
    temp = {k: i for k, i in temp.items() if k not in remove}

if assessment == 'optimal_pp':
    remove = ['No friction', 'Second best subsidies', 'Baseline Package']
    temp = {k: i for k, i in temp.items() if k not in remove}

    temp_package = {data.loc[k, 'Group']: item for k, item in dict_output_subset.items()}
    temp_package = {k: i for k, i in temp_package.items() if k in remove + [reference]}
    
    _ = indicator_policies(temp_package, folder, config_policies, reference=reference, figure=True, order_scenarios=remove)

_ = indicator_policies(temp, folder, config_policies, reference=reference, figure=True, order_scenarios=order_scenarios)